In [11]:
!pip install biopython
!pip install -q condacolab
!conda install -c bioconda seqkit -y


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Channels:
 - bioconda
 - conda-forge
 - defaults
Platform: linux-64
Solving environment: done

# All requested packages already installed.



In [8]:
import subprocess
import re
import requests
from Bio import SeqIO
import json

In [25]:
class MyFastaParser:

    def __init__(self, fasta_file):

        self.fasta_file = fasta_file

    def get_uniprot(self, accession):

        endpoint = f"https://rest.uniprot.org/uniprotkb/{accession}"
        http_function = requests.get
        http_args = {'params': {'accession': accession}}
        return http_function(endpoint, **http_args)


    def get_ensembl(self, id):

        endpoint = f"https://rest.ensembl.org/lookup/id/{id}"
        http_function = requests.get
        http_args = {'headers': {'Content-Type': 'application/json'}}

        return http_function(endpoint, **http_args)


    def uniprot_parse_response(self, resp):

        if not resp.ok:
            return f"HTTP error: {resp.status_code}"

        try:
            resp = resp.json()
        except ValueError:
            raise ValueError("Invalid JSON response")

        acc = resp.get('primaryAccession')
        if not acc:
            raise ValueError("Missing primaryAccession in response")

        species = resp.get('organism', {}).get('scientificName')
        gene = resp.get('genes', [])
        seq_dict = resp.get('sequence', {})

        return {
            acc: {
                'organism': species,
                'geneInfo': gene,
                'sequenceInfo': {
                    'value': seq_dict.get('value'),
                    'length': seq_dict.get('length'),
                    'molWeight': seq_dict.get('molWeight'),
                    'crc64': seq_dict.get('crc64'),
                    'md5': seq_dict.get('md5')
                },
                'type': 'protein'
            }
        }

    def ensembl_parse_response(self, resp):

        if not resp.ok:
            return f"HTTP error: {resp.status_code}"

        try:
            resp = resp.json()
        except ValueError:
            raise ValueError("Invalid JSON response from Ensembl")

        gene_id = resp.get('id')
        if not gene_id:
            raise ValueError("Missing 'id' field in Ensembl response")

        output = {
            gene_id: {
                'object_type': resp.get('object_type', None),
                'species': resp.get('species', None),
                'assembly_name': resp.get('assembly_name', None),
                'biotype': resp.get('biotype', None),
                'display_name': resp.get('display_name', None),
                'id': gene_id,
                'db_type': resp.get('db_type', None),
                'description': resp.get('description', None),
                'source': resp.get('source', None),
                'canonical_transcript': resp.get('canonical_transcript', None)
            }
        }

        return output


    def run_seqkit(self):

        try:
            result = subprocess.run(
                ("seqkit", "stats", "-a", self.fasta_file),
                capture_output=True,
                text=True,
                check=True
            )

            return result.stdout

        except FileNotFoundError:
            return {
                "error": "seqkit is not installed or not in PATH",
            }

        except subprocess.CalledProcessError as e:
            return {
                "error": "seqkit execution failed",
                "details": e.stderr
            }

    def seqkit_stats(self) -> dict:

        try:
            result = subprocess.run(
                ['seqkit', 'stats', '-a', '-T', self.fasta_file],
                capture_output=True,
                text=True,
                check=True
            )
        except FileNotFoundError:
            return {"error": "seqkit is not installed or not in PATH"}
        except subprocess.CalledProcessError as e:
            return {"error": e.stderr}

        lines = result.stdout.strip().split('\n')
        if len(lines) < 2:
            return {'error': 'File structure breaks seqkit parsing.'}

        headers = lines[0].split('\t')
        values = lines[1].split('\t')

        stat_info = dict(zip(headers, values))
        stat_info = {k: stat_info.get(k, '0') for k in headers}

        self.seq_type = stat_info.get('type', 'Unknown')
        num_seqs = int(stat_info.get('num_seqs', 0))

        output = {
            'fasta_seqkit_stat_info': stat_info,
            'fasta_type': self.seq_type,
            'fasta_num_seqs': num_seqs
        }
        return output
        
    def extract_id(self, description):
        
        if self.seq_type == "Protein":
            match = re.search(r'\b[OPQ][0-9][A-Z0-9]{3}[0-9]\b', description)
            if match:
                return match.group(0)

        elif self.seq_type == "DNA":
            match = re.search(r'\bENS[A-Z]+\d+\b', description)
            if match:
                return match.group(0)

        return None

    def biopython_parser(self, stats):
        
        from Bio import SeqIO

        results = {}
        warning_flag = False

        for record in SeqIO.parse(self.fasta_file, "fasta"):
            seq_id =  self.extract_id(record.id)
            seq_description = record.description
            seq = str(record.seq)
            
            if seq_id is None:
                warning_flag = True
                continue

            db_info = {}
            db_name = None

            if self.seq_type == "Protein":
                resp = self.get_uniprot(seq_id)
                parsed = self.uniprot_parse_response(resp)
                if 'error' in parsed:
                    print(seq_id, parsed)
                    continue
                db_info = parsed.get(seq_id, {})
                db_name = "uniprot"

            elif self.seq_type == "DNA":
                resp = self.get_ensembl(seq_id)
                parsed = self.ensembl_parse_response(resp)
                if 'error' in parsed:
                    print(seq_id, parsed)
                    continue
                db_info = parsed.get(seq_id, {})
                db_name = "ensembl"

            if "DB_name" not in results:
                results["DB_name"] = db_name

            results[f"file_info_{seq_id}"] = {
                "description": seq_description,
                "sequence": seq
            }

            results[f"database_info_{seq_id}"] = db_info

        if warning_flag:
            results["WARNING"] = {"No ID match found."}

        return results
      
    def show_output(self, output, indent=0):
        for key, value in output.items():
            print('\t' * indent + str(key))
            if isinstance(value, dict):
                self.show_output(value, indent + 1)
            else:
                print('\t' * (indent + 1) + str(value))

In [26]:
parser = MyFastaParser('test_file.fasta')
stats = parser.seqkit_stats()
print(json.dumps(stats, indent = 2))

{
  "fasta_seqkit_stat_info": {
    "file": "test_file.fasta",
    "format": "FASTA",
    "type": "Protein",
    "num_seqs": "2",
    "sum_len": "456",
    "min_len": "29",
    "avg_len": "228.0",
    "max_len": "427",
    "Q1": "29",
    "Q2": "228",
    "Q3": "427",
    "sum_gap": "0",
    "N50": "427",
    "N50_num": "1",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "AvgQual": "0.00",
    "GC(%)": "0.00",
    "sum_n": "0"
  },
  "fasta_type": "Protein",
  "fasta_num_seqs": 2
}


In [27]:
biopython = parser.biopython_parser(stats)
parser.show_output(biopython)

DB_name
	uniprot
file_info_P11473
	description
		sp|P11473|VDR_HUMAN Vitamin D3 receptor OS=Homo sapiens OX=9606 GN=VDR PE=1 SV=1
	sequence
		MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSSDMMDSSSFSNLDLSEEDSDDPSVTLELSQLSMLPHLADLVSYSIQKVIGFAKMIPGFRDLTSEDQIVLLKSSAIEVIMLRSNESFTMDDMSWTCGNQDYKYRVSDVTKAGHSLELIEPLIKFQVGLKKLNLHEEEHVLLMAICIVSPDRPGVQDAALIEAIQDRLSNTLQTYIRCRHPPPGSHLLYAKMIQKLADLRSLNEEHSKQYRCLSFQPECSMKLTPLVLEVFGNEIS
database_info_P11473
	organism
		Homo sapiens
	geneInfo
		[{'geneName': {'evidences': [{'evidenceCode': 'ECO:0000312', 'source': 'HGNC', 'id': 'HGNC:12679'}], 'value': 'VDR'}, 'synonyms': [{'value': 'NR1I1'}]}]
	sequenceInfo
		value
			MEAMAASTSLPDPGDFDRNVPRICGVCGDRATGFHFNAMTCEGCKGFFRRSMKRKALFTCPFNGDCRITKDNRRHCQACRLKRCVDIGMMKEFILTDEEVQRKREMILKRKEEEALKDSLRPKLSEEQQRIIAILLDAHHKTYDPTYSDFCQFRPPVRVNDGGGSHPSRPNSRHTPSFSGDSSSSCSDHCITSS

In [28]:
parser2 = MyFastaParser('ensembl_download_1.fasta')
stats = parser2.seqkit_stats()
print(json.dumps(stats, indent = 2))

{
  "fasta_seqkit_stat_info": {
    "file": "ensembl_download_1.fasta",
    "format": "FASTA",
    "type": "DNA",
    "num_seqs": "6",
    "sum_len": "86",
    "min_len": "9",
    "avg_len": "14.3",
    "max_len": "23",
    "Q1": "10",
    "Q2": "14",
    "Q3": "17",
    "sum_gap": "0",
    "N50": "16",
    "N50_num": "3",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "AvgQual": "0.00",
    "GC(%)": "45.35",
    "sum_n": "0"
  },
  "fasta_type": "DNA",
  "fasta_num_seqs": 6
}


In [32]:
biopython = parser2.biopython_parser(stats)
parser2.show_output(biopython)

ENSDART0000018940031 HTTP error: 400
DB_name
	ensembl
file_info_ENSMUST00000196221
	description
		ENSMUST00000196221.2 cds chromosome:GRCm39:14:54350925:54350933:1 gene:ENSMUSG00000096749.3 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd1 description:T cell receptor delta diversity 1 [Source:MGI Symbol;Acc:MGI:4439547]
	sequence
		ATGGCATAT
database_info_ENSMUST00000196221
	object_type
		Transcript
	species
		mus_musculus
	assembly_name
		GRCm39
	biotype
		TR_D_gene
	display_name
		Trdd1-202
	id
		ENSMUST00000196221
	db_type
		core
	description
		None
	source
		havana
	canonical_transcript
		None
file_info_ENSMUST00000177564
	description
		ENSMUST00000177564.2 cds chromosome:GRCm39:14:54359683:54359698:1 gene:ENSMUSG00000096176.2 gene_biotype:TR_D_gene transcript_biotype:TR_D_gene gene_symbol:Trdd2 description:T cell receptor delta diversity 2 [Source:MGI Symbol;Acc:MGI:4439546]
	sequence
		ATCGGAGGGATACGAG
database_info_ENSMUST00000177564
	object_type
		Transcript

In [33]:
parser3 = MyFastaParser('ensembl_download_2.fasta')
stats = parser3.seqkit_stats()
print(json.dumps(stats, indent = 2))

{
  "error": "File structure breaks seqkit parsing."
}


In [34]:
parser4 = MyFastaParser('uniprot_download.fasta')
stats = parser4.seqkit_stats()
print(json.dumps(stats, indent = 2))

{
  "fasta_seqkit_stat_info": {
    "file": "uniprot_download.fasta",
    "format": "FASTA",
    "type": "Protein",
    "num_seqs": "7",
    "sum_len": "3861",
    "min_len": "180",
    "avg_len": "551.6",
    "max_len": "1382",
    "Q1": "429",
    "Q2": "441",
    "Q3": "500",
    "sum_gap": "0",
    "N50": "468",
    "N50_num": "3",
    "Q20(%)": "0",
    "Q30(%)": "0",
    "AvgQual": "0.00",
    "GC(%)": "0.00",
    "sum_n": "0"
  },
  "fasta_type": "Protein",
  "fasta_num_seqs": 7
}


In [35]:
biopython = parser4.biopython_parser(stats)
parser4.show_output(biopython)

DB_name
	uniprot
file_info_Q9R1A7
	description
		sp|Q9R1A7|NR1I2_RAT Nuclear receptor subfamily 1 group I member 2 OS=Rattus norvegicus OX=10116 GN=Nr1i2 PE=2 SV=1
	sequence
		MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLLPHLADVSTYMFKGVINFAKVISHFRELPIEDQISLLKGATFEMCILRFNTMFDTETGTWECGRLAYCFEDPNGGFQKLLLDPLMKFHCMLKKLQLREEEYVLMQAISLFSPDRPGVVQRSVVDQLQERFALTLKAYIECSRPYPAHRFLFLKIMAVLTELRSINAQQTQQLLRIQDTHPFATPLMQELFSSTDG
database_info_Q9R1A7
	organism
		Rattus norvegicus
	geneInfo
		[{'geneName': {'value': 'Nr1i2'}, 'synonyms': [{'value': 'Pxr'}]}]
	sequenceInfo
		value
			MRPEERWNHVGLVQREEADSVLEEPINVDEEDGGLQICRVCGDKANGYHFNVMTCEGCKGFFRRAMKRNVRLRCPFRKGTCEITRKTRRQCQACRLRKCLESGMKKEMIMSDAAVEQRRALIKRKKREKIEAPPPGGQGLTEEQQALIQELMDAQMQTFDTTFSHFKDFRLPAVFHSDCELPEVLQASLLEDPATWSQIMKDSVPMKISVQLRGEDGSIWNYQPPSKSDGKEIIPLL